# 01 · Explore job postings

Run each cell with **Shift+Enter**. Start with the offline sample; switch to the live API once your key is in `.env`.

The point of this notebook is to poke at data interactively, the way you'd use the C# Interactive window or LINQPad.

In [ ]:
from jobscout import jsearch, store

jobs = jsearch.load_sample()          # offline
# jobs = jsearch.search("technical program manager in Florida")   # live API

len(jobs), jobs[0]

A notebook shows the value of the last line in a cell, so no `print` needed. Note how `jobs[0]` prints nicely: that's the `__repr__` that `@dataclass` generated.

In [ ]:
df = store.to_frame(jobs)
df.head()

## pandas as SQL you can step through

Each line below has a SQL equivalent in the comment.

In [ ]:
# SELECT title, employer, city FROM df WHERE is_remote = 1
df.loc[df.is_remote, ["title", "employer", "city"]]

In [ ]:
# SELECT employer, COUNT(*) AS n FROM df GROUP BY employer ORDER BY n DESC
df.groupby("employer").size().sort_values(ascending=False)

In [ ]:
# WHERE title LIKE '%Program Manager%'
df[df.title.str.contains("Program Manager", case=False)][["title", "employer", "min_salary", "max_salary"]]

## A first crude 'score'

Before we bring in Claude, a keyword score. It's a baseline: later you can compare it against the LLM score and see where they disagree.

In [ ]:
MY_KEYWORDS = ["azure", ".net", "fda", "iso 13485", "agile", "safe", "stakeholder"]

def keyword_score(text: str) -> int:
    text = text.lower()
    return sum(kw in text for kw in MY_KEYWORDS)   # True counts as 1

df["kw_score"] = df.description.apply(keyword_score)
df.sort_values("kw_score", ascending=False)[["title", "employer", "kw_score"]].head(10)

## Try it yourself

1. Add a column `salary_mid` = average of min and max. (Hint: arithmetic on columns works element-wise.)
2. Show only Florida jobs posted in the last 7 days.
3. Change `keyword_score` so FDA/ISO terms count double.
4. Save your filtered result with `df.to_csv(...)` and open it in Excel.